# 02 — Modeling walkthrough
Runs the full training pipeline and inspects the leaderboard + SHAP.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from src.pipeline import run
result = run()
result['leaderboard']

In [ ]:
import joblib, pandas as pd
from src.config import MODELS_DIR, PROCESSED_CSV
art = joblib.load(MODELS_DIR / 'artifacts.joblib')
df = pd.read_csv(PROCESSED_CSV)
print('Best model:', art['model_name'])

In [ ]:
from src import preprocessing, feature_engineering
from src.explainability import shap_values_for
import pandas as pd
eng = feature_engineering.add_engineered_features(preprocessing.clean(df))
X = eng.drop(columns=[c for c in ['Churn','customerID','cluster','segment'] if c in eng.columns])
Xt = art['preprocessor'].transform(X)
Xt_df = pd.DataFrame(Xt, columns=art['feature_names'])
sv, base, sample = shap_values_for(art['model'], Xt_df.sample(300, random_state=1))
import shap, matplotlib.pyplot as plt
shap.summary_plot(sv, sample, show=True)